In [ ]:
import os
import pandas as pd
from pathlib import Path

# Root directory of your dataset
DATA_DIR = Path("CUB_200_2011_Subset20classes").resolve()
# Path to the image folder
IMAGE_DIR = DATA_DIR / "images"

####################################
## Step 1: Load Metadata Files
####################################

# images.txt: maps image_id → relative file path
images_df = pd.read_csv(
    os.path.join(DATA_DIR, "images.txt"),
    sep=" ",
    names=["image_id", "image_path"]
)

# image_class_labels.txt: maps image_id → class label
labels_df = pd.read_csv(
    os.path.join(DATA_DIR, "image_class_labels.txt"),
    sep=" ",
    names=["image_id", "class_id"]
)

# bounding_boxes.txt: provides object localisation (bird region)
bbox_df = pd.read_csv(
    os.path.join(DATA_DIR, "bounding_boxes.txt"),
    sep=" ",
    names=["image_id", "x", "y", "width", "height"]
)

# train_test_split.txt: indicates whether image is training or testing
split_df = pd.read_csv(
    os.path.join(DATA_DIR, "train_test_split.txt"),
    sep=" ",
    names=["image_id", "is_train"]
)

####################################
## Step 2: Merge All Information
####################################

# merging into one dataframe for easier handling

data = images_df.merge(labels_df, on="image_id")
data = data.merge(bbox_df, on="image_id")
data = data.merge(split_df, on="image_id")


####################################
## Step 3: Remap class labels to 0–19
####################################
# Get unique class IDs (e.g., [3, 7, 15, ...]), sorting by class id for consistency
# Data is sorted by class_id to ensure consistent mapping across runs
unique_classes = sorted(data["class_id"].unique())
# Create mapping → key value pairs are switched around to value key pairs
class_mapping = {old: new for new, old in enumerate(unique_classes)}
# Apply mapping, so that class_id is remapped to 0–19
data["label"] = data["class_id"].map(class_mapping)


####################################
## Step 4: Build Full Image Paths
####################################
# Construct full image paths by joining the base image directory with the relative paths from images.txt

def fix_path(rel_path):
    # Ensure rel_path is a string and stripped
    rel_path = str(rel_path).strip()
    
    # 2. IMAGE_DIR (Path) / rel_path (str) = a new Path object
    full_path = (IMAGE_DIR / rel_path).resolve()
    
    # Return as a standard Windows string (using backslashes \)
    return str(full_path)

data["full_path"] = data["image_path"].apply(fix_path)

####################################
## Step 5: Split into Training and Testing Sets
####################################
train_df = data[data["is_train"] == 1].reset_index(drop=True)
test_df = data[data["is_train"] == 0].reset_index(drop=True)

train_paths = train_df["full_path"].values
train_labels = train_df["label"].values
train_bboxes = train_df[["x", "y", "width", "height"]].values

test_paths = test_df["full_path"].values
test_labels = test_df["label"].values
test_bboxes = test_df[["x", "y", "width", "height"]].values

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))
print("Number of classes:", data["label"].nunique())
print(test_paths[:5])  # Print first 5 test image paths

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [2]:
from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.transform import resize
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from PIL import Image
import numpy as np

# This function extracts HOG feature from an image
def extract_hog(image_path):
    # Load image and convert to RGB
    image = Image.open(image_path).convert("RGB")
    # Resize to fixed size for consistent feature extraction
    image = resize(np.array(image), (128, 128))
    # Convert to grayscale (HOG works on intensity gradients)
    gray = rgb2gray(image)
    # Extract HOG features (edge/shape descriptors)
    features = hog(
    gray,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2)
    )
    return features

# This function builds the dataset (X, y)
def build_dataset(paths, labels):
    X, y = [], []
    for p, l in zip(paths, labels):
        # Extract feature for each image
        X.append(extract_hog(p))
        y.append(l)
    return np.array(X), np.array(y)

# Build training and testing feature sets
X_train, y_train = build_dataset(train_paths, train_labels)
X_test, y_test = build_dataset(test_paths, test_labels)
# Train SVM classifier (linear kernel works well for HOG)
model = SVC(kernel="linear")
model.fit(X_train, y_train)
# Predict on test data
y_pred = model.predict(X_test)
# Evaluate performance
print("Experiment 1 (Whole Image + HOG + SVM) Accuracy:", accuracy_score(y_test, y_pred))


Experiment 1 (Whole Image + HOG + SVM) Accuracy: 0.16143497757847533


In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 20

def load_image(path, label):
    # Read image file
    img = tf.io.read_file(path)
    # Decode JPEG → tensor
    img = tf.image.decode_jpeg(img, channels=3)
    # Resize to CNN input size
    img = tf.image.resize(img, IMG_SIZE)
    # Normalize pixel values to [0,1]
    img = img / 255.0
    return img, label

# Build TensorFlow dataset:
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.map(load_image).shuffle(500).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(load_image).batch(BATCH_SIZE)

def cnn_scratch():
    # Define simple CNN architecture
    model = models.Sequential([
        layers.Input((224,224,3)),
        # Learn low-level features (edges, textures)
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(),
        # Learn more complex patterns
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        # Learn higher-level object structures
        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(),
        # Flatten feature maps to vector
        layers.Flatten(),
        # Fully connected layer for classification
        layers.Dense(128, activation='relu'),
        # Output layer (20 bird classes)
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    # Compile model
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = cnn_scratch()
# Train CNN
model.fit(train_ds, epochs=10)
# Evaluate performance
print("Experiment 2A (Whole Image + CNN from Scratch) Accuracy:", model.evaluate(test_ds)[1])

Epoch 1/10


InvalidArgumentError: Graph execution error:

Detected at node ReadFile defined at (most recent call last):
<stack traces unavailable>
Error in user-defined function passed to MapDataset:8 transformation with iterator: Iterator::Root::Prefetch::BatchV2::Shuffle::ParallelMapV2: NewRandomAccessFile failed to Create/Open: \\Studentfiles.win.canberra.edu.au\Homes$\u3263629\My Documents\comp_vision\CUB_200_2011_Subset20classes\images\001.Black_footed_Albatross\Black_Footed_Albatross_0046_18.jpg : The filename, directory name, or volume label syntax is incorrect.
; no protocol option
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_3962]